# Bronze Ingest — users_2021
**Proyecto 2 FHBD | Capa Bronze — Ejecución Manual**

Equivalente manual del Task 1 del DAG.

Usa `dlt` (Data Load Tool) para descargar `users_2021` desde ClickHouse público
y guardarlo en MinIO como Parquet con **OVERWRITE**.

**Ruta destino:** `s3://bronze/users/2021/users_2021.parquet`

---
⚠️ Ejecutar este notebook si el Task 1 del DAG falla.

## 1. Instalación de dependencias

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'dlt[filesystem]==0.5.4',
    'clickhouse-connect==0.7.19',
    'boto3==1.35.0',
    's3fs==2024.6.1',
    'pyarrow==17.0.0'
])
print('✅ Dependencias instaladas')

## 2. Configuración

In [ ]:
import os

# MinIO
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT',  'http://proyecto2-minio:9000')
MINIO_ACCESS   = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET   = os.getenv('MINIO_SECRET_KEY', 'minioadmin123')
BRONZE_BUCKET  = os.getenv('MINIO_BUCKET_BRONZE', 'bronze')

# ClickHouse público
CH_HOST     = os.getenv('CH_HOST',     'clickhouse.clickhouse.com')
CH_PORT     = int(os.getenv('CH_PORT', '443'))
CH_USER     = os.getenv('CH_USER',     'play')
CH_PASSWORD = os.getenv('CH_PASSWORD', '')
CH_DB       = os.getenv('CH_DB',       'stackoverflow')

MAX_ROWS       = int(os.getenv('MAX_ROWS', '100000'))
PIPELINE_YEAR  = 2021

print(f'MinIO   : {MINIO_ENDPOINT}')
print(f'Bucket  : {BRONZE_BUCKET}')
print(f'CH Host : {CH_HOST}:{CH_PORT}')
print(f'Año     : {PIPELINE_YEAR}')
print(f'Max rows: {MAX_ROWS:,}')

## 3. Verificar conexión a ClickHouse

In [ ]:
import clickhouse_connect

ch = clickhouse_connect.get_client(
    host=CH_HOST, port=CH_PORT,
    username=CH_USER, password=CH_PASSWORD,
    database=CH_DB, secure=(CH_PORT == 443)
)

# Contar registros disponibles
count = ch.query_df(f'SELECT COUNT(*) AS cnt FROM users WHERE toYear(CreationDate) = {PIPELINE_YEAR}')
print(f'✅ Conexión OK')
print(f'   Registros disponibles en users {PIPELINE_YEAR}: {count.iloc[0]["cnt"]:,}')

## 4. Definir pipeline dlt

In [ ]:
import dlt

@dlt.resource(
    name='users_2021',
    write_disposition='replace',  # OVERWRITE — comportamiento Bronze
    primary_key='Id'
)
def users_2021_source():
    """Resource dlt: descarga users 2021 desde ClickHouse."""
    df = ch.query_df(f"""
        SELECT
            Id, Reputation, CreationDate, DisplayName,
            LastAccessDate, Views, UpVotes, DownVotes,
            Location, AccountId
        FROM users
        WHERE toYear(CreationDate) = {PIPELINE_YEAR}
        LIMIT {MAX_ROWS}
    """)
    print(f'  Filas descargadas: {len(df):,}')
    yield df

print('✅ Resource dlt definido')

## 5. Ejecutar pipeline dlt → MinIO

In [ ]:
# Configurar credenciales S3/MinIO para dlt
os.environ['DESTINATION__FILESYSTEM__BUCKET_URL'] = (
    f's3://{BRONZE_BUCKET}/users/{PIPELINE_YEAR}'
)
os.environ['DESTINATION__FILESYSTEM__CREDENTIALS__AWS_ACCESS_KEY_ID']     = MINIO_ACCESS
os.environ['DESTINATION__FILESYSTEM__CREDENTIALS__AWS_SECRET_ACCESS_KEY'] = MINIO_SECRET
os.environ['DESTINATION__FILESYSTEM__CREDENTIALS__ENDPOINT_URL']          = MINIO_ENDPOINT
os.environ['DESTINATION__FILESYSTEM__CREDENTIALS__REGION_NAME']           = 'us-east-1'

pipeline = dlt.pipeline(
    pipeline_name='bronze_users_2021',
    destination='filesystem',
    dataset_name=f'users_{PIPELINE_YEAR}',
    pipelines_dir='/tmp/dlt_pipelines',
)

print('Ejecutando pipeline dlt...')
load_info = pipeline.run(users_2021_source())
print(load_info)
print('✅ Pipeline dlt completado')

## 6. Renombrar archivo al path estándar

In [ ]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS,
    aws_secret_access_key=MINIO_SECRET,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1',
)

prefix = f'users_{PIPELINE_YEAR}/users_2021/'
response = s3.list_objects_v2(Bucket=BRONZE_BUCKET, Prefix=prefix)

parquet_files = [
    obj for obj in response.get('Contents', [])
    if obj['Key'].endswith('.parquet')
]

if parquet_files:
    source_key = parquet_files[-1]['Key']
    target_key = f'users/{PIPELINE_YEAR}/users_{PIPELINE_YEAR}.parquet'
    
    s3.copy_object(
        Bucket=BRONZE_BUCKET,
        CopySource={'Bucket': BRONZE_BUCKET, 'Key': source_key},
        Key=target_key,
    )
    s3.delete_object(Bucket=BRONZE_BUCKET, Key=source_key)
    print(f'✅ Archivo estandarizado: s3://{BRONZE_BUCKET}/{target_key}')
else:
    print('⚠️  No se encontraron archivos parquet del pipeline dlt')

## 7. Verificación final

In [ ]:
import pandas as pd, io

key = f'users/{PIPELINE_YEAR}/users_{PIPELINE_YEAR}.parquet'
obj = s3.get_object(Bucket=BRONZE_BUCKET, Key=key)
df  = pd.read_parquet(io.BytesIO(obj['Body'].read()))

print(f'✅ Verificación Bronze users_{PIPELINE_YEAR}')
print(f'   Ruta  : s3://{BRONZE_BUCKET}/{key}')
print(f'   Filas : {len(df):,}')
print(f'   Cols  : {list(df.columns)}')
print()
df.head(5)